# mis_analytics

> DataFrame utilities, pipeline tracking, and optional SAP data dictionary helpers.

`mis_analytics` helps prepare supply chain and manufacturing data. Clean columns and resample time series, inspect pandas and eager Polars pipelines, and use deSAPher to look up SAP fields and convert SAP extracts.

## Installation

Requires Python 3.11 or later. Install the DataFrame utilities and pipeline tracking:

```sh
pip install mis_analytics
```

For SAP functionality, install the `desapher` extra:

```sh
pip install "mis_analytics[desapher]"
```

The extra adds `httpx`, `beautifulsoup4`, and `lxml`. The base package includes pandas, NumPy, and Polars. Import `mis_analytics.desapher` only when its extra dependencies are installed.

To install the latest code from GitHub, including the extra:

```sh
pip install "mis_analytics[desapher] @ git+https://github.com/MIS-Analytics/mis_analytics.git"
```

Omit `[desapher]` for the base package.

## DataFrame utilities

In [ ]:
import pandas as pd
import polars as pl
from mis_analytics.core import clean_col_names
from mis_analytics.etl import track

In [ ]:
orders = pd.DataFrame({'Order ID': [101, 102, 103], 'Quantity': [5, 10, 15]})
orders = clean_col_names(orders)
orders

,order_id,quantity
0,101,5
1,102,10
2,103,15


## Pipeline tracking

Decorate a function with `@track` and chain it with `.pipe()`. Set `vrbs=True` on each step to print its name, docstring, elapsed time, shape changes, and input/output samples. Logging is off by default. Both pandas and eager Polars DataFrames are supported; each transformation must support the DataFrame it receives.

In [ ]:
@track
def take_rows(df, n):
    """Keep the first n rows."""
    return df.head(n)

In [ ]:
pandas_preview = orders.pipe(take_rows, 2, vrbs=True).pipe(take_rows, 1, vrbs=True)

*************** take_rows ***************
'''Keep the first n rows.'''


Total Time: 691 µs

Start: 2026-09-16 11:42:40.985041
  End: 2026-09-16 11:42:40.985732

Input DataFrame:
head:
   order_id  quantity
0       101         5
1       102        10
2       103        15
sample:
   order_id  quantity
1       102        10
2       103        15
0       101         5
tail:
   order_id  quantity
0       101         5
1       102        10
2       103        15

        Input:   3 rows, 2 cols
                     ↓       ↓
        Diff:   -1 rows, 0 cols
                     ↓       ↓
        Output:  2 rows, 2 cols
        
Output DataFrame:
head:
   order_id  quantity
0       101         5
1       102        10
sample:
   order_id  quantity
0       101         5
1       102        10
tail:
   order_id  quantity
0       101         5
1       102        10


*************** take_rows ***************
'''Keep the first n rows.'''


Total Time: 217 µs

Start: 2026-09-16 11:42:40.989443
  En

In [ ]:
polars_orders = pl.DataFrame({'order_id': [101, 102, 103], 'quantity': [5, 10, 15]})
polars_preview = polars_orders.pipe(take_rows, 2, vrbs=True).pipe(take_rows, 1, vrbs=True)

*************** take_rows ***************
'''Keep the first n rows.'''


Total Time: 257 µs

Start: 2026-09-16 11:42:58.817200
  End: 2026-09-16 11:42:58.817457

Input DataFrame:
head:
shape: (3, 2)
┌──────────┬──────────┐
│ order_id ┆ quantity │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 101      ┆ 5        │
│ 102      ┆ 10       │
│ 103      ┆ 15       │
└──────────┴──────────┘
sample:
shape: (3, 2)
┌──────────┬──────────┐
│ order_id ┆ quantity │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 101      ┆ 5        │
│ 102      ┆ 10       │
│ 103      ┆ 15       │
└──────────┴──────────┘
tail:
shape: (3, 2)
┌──────────┬──────────┐
│ order_id ┆ quantity │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 101      ┆ 5        │
│ 102      ┆ 10       │
│ 103      ┆ 15       │
└──────────┴──────────┘

        Input:   3 rows, 2 cols
                     ↓       ↓
        Diff:   -1 rows, 0 cols
               

## SAP helpers with deSAPher

With the `desapher` extra installed, fetch SAP table metadata from sapdatasheet.org and use it to convert types and rename fields in pandas extracts. Fetching metadata requires internet access.

```python
from mis_analytics.desapher import get_sap_tables_structure, convert_sap_types, rename_sap_columns

sap_sheet = get_sap_tables_structure(['MARA'])
materials = pd.DataFrame({'MATNR': ['000000000000000101']})
if sap_sheet is not None:
    materials = materials.pipe(convert_sap_types, sap_sheet).pipe(rename_sap_columns, sap_sheet)
```

`get_sap_table_description` also retrieves a table's description.

## Development

From a local checkout, install the development group and SAP extra:

```sh
uv sync --group dev --extra desapher
```

Edit the source notebooks under `nbs/`, then export the modules. Regenerate the README after editing `nbs/index.ipynb`:

```sh
uv run nbdev-export
uv run nbdev-test --path nbs/index.ipynb
uv run nbdev-readme
```

[API documentation](https://MIS-Analytics.github.io/mis_analytics/) · [Source code](https://github.com/MIS-Analytics/mis_analytics)